# KishoLens ML & NLP Feature Extraction Prototype

This notebook prototypes the core NLP processing and feature extraction modules of KishoLens. It loads crawled chapter text from SQLite (`data/kisholens.db`), computes language-specific stylistic pacing features (dialogue ratio, vocabulary diversity, punctuation density, sentence length, and pacing metrics), and aggregates statistics by novel and ingestion source.

In [ ]:
import os
import re
import unicodedata
from typing import Optional, List, Dict, Any
from sqlmodel import SQLModel, Field, Session, create_engine, select
import pandas as pd

In [ ]:
# Declare SQLModel tables (with extend_existing=True for notebook rerun compatibility)

class Novel(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    author: str
    source: str

class Chapter(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    novel_id: int = Field(foreign_key="novel.id")
    chapter_number: int
    title: str
    text_ja: str
    text_en: str

In [ ]:
# Core NLP Feature Extraction Modules

def compute_type_token_ratio(tokens: List[str]) -> float:
    """Computes Type-Token Ratio (vocabulary diversity)."""
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def extract_english_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from English text:
    - Token (word) count
    - Sentence count
    - Average sentence length
    - Dialogue ratio (dialogue lines / total lines)
    - Vocabulary diversity (Type-Token Ratio)
    - Punctuation density
    """
    if not text:
        return {"word_count": 0, "sentence_count": 0, "avg_sentence_len": 0.0, "dialogue_ratio": 0.0, "ttr": 0.0, "punc_density": 0.0}
        
    # Words
    words = re.findall(r'\b\w+\b', text.lower())
    word_count = len(words)
    
    # Sentences split by terminal punctuation
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    sentence_count = len(sentences)
    avg_sentence_len = word_count / sentence_count if sentence_count > 0 else 0.0
    
    # Dialogue lines starting with quote punctuation
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('"') or l.startswith("'") or l.startswith('“') or l.startswith('”')]
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    
    # Type-token ratio
    ttr = compute_type_token_ratio(words)
    
    # Punctuation density
    punctuations = re.findall(r'[.,\/#!$%\^&\*;:{}=\-_`~()?"\']', text)
    punc_count = len(punctuations)
    char_count = len(text)
    punc_density = punc_count / char_count if char_count > 0 else 0.0
    
    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density
    }

def extract_japanese_features(text: str) -> Dict[str, Any]:
    """
    Extracts stylistic features from Japanese text:
    - Character count (acting as token count for baseline)
    - Sentence count
    - Average sentence length (characters per sentence)
    - Dialogue ratio (lines starting with 「 or 『)
    - Vocabulary diversity (Character TTR as a proxy)
    - Punctuation density
    """
    if not text:
        return {"char_count": 0, "sentence_count": 0, "avg_sentence_len": 0.0, "dialogue_ratio": 0.0, "ttr": 0.0, "punc_density": 0.0}
        
    # Clean whitespaces for token count proxy
    chars = [c for c in text if not c.isspace()]
    char_count = len(chars)
    
    # Sentences split by terminal punctuation
    sentences = [s.strip() for s in re.split(r'[。！？]+', text) if s.strip()]
    sentence_count = len(sentences)
    avg_sentence_len = char_count / sentence_count if sentence_count > 0 else 0.0
    
    # Dialogue lines starting with Japanese open quotes
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    dialogue_lines = [l for l in lines if l.startswith('「') or l.startswith('『')]
    dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    
    # Vocabulary diversity proxy
    ttr = compute_type_token_ratio(chars)
    
    # Punctuation density
    punctuations = re.findall(r'[、。！？「」『』（）―…ー・]', text)
    punc_count = len(punctuations)
    total_chars = len(text)
    punc_density = punc_count / total_chars if total_chars > 0 else 0.0
    
    return {
        "char_count": char_count,
        "sentence_count": sentence_count,
        "avg_sentence_len": avg_sentence_len,
        "dialogue_ratio": dialogue_ratio,
        "ttr": ttr,
        "punc_density": punc_density
    }

In [ ]:
# Load Database and Process Chapters

db_path = "data/kisholens.db" if os.path.exists("data/kisholens.db") else "../data/kisholens.db"
engine = create_engine(f"sqlite:///{db_path}")

features_list = []

with Session(engine) as session:
    # Load all novels and map their IDs to metadata
    novels = session.exec(select(Novel)).all()
    novels_map = {n.id: n for n in novels}
    
    # Load all chapters
    chapters = session.exec(select(Chapter)).all()
    print(f"Processing {len(chapters)} chapters...")
    
    for ch in chapters:
        novel = novels_map.get(ch.novel_id)
        if not novel:
            continue
            
        # Extract features for available text fields
        row = {
            "novel_title": novel.title,
            "author": novel.author,
            "source": novel.source,
            "chapter_num": ch.chapter_number,
            "chapter_title": ch.title
        }
        
        if ch.text_en:
            en_feat = extract_english_features(ch.text_en)
            row.update({f"en_{k}": v for k, v in en_feat.items()})
            
        if ch.text_ja:
            ja_feat = extract_japanese_features(ch.text_ja)
            row.update({f"ja_{k}": v for k, v in ja_feat.items()})
            
        features_list.append(row)

df = pd.DataFrame(features_list)
df.head()

In [ ]:
# aggregate Stylistic Stats by Novel Source

print("Style Aggregates by Source Platform:")
group_cols = ["source"]

# Select numerical columns for aggregation
num_cols = [c for c in df.columns if c.startswith("en_") or c.startswith("ja_")]
agg_df = df.groupby(group_cols)[num_cols].mean()
agg_df

In [ ]:
# Stylistic pacing comparisons (English texts)

en_cols = [c for c in df.columns if c.startswith("en_")]
if en_cols:
    print("English Style Aggregates by Novel:")
    novel_en = df.groupby(["novel_title", "source"])[en_cols].mean()
    print(novel_en.to_string())
else:
    print("No English style columns available.")

In [ ]:
# Stylistic pacing comparisons (Japanese texts)

ja_cols = [c for c in df.columns if c.startswith("ja_")]
if ja_cols:
    print("\nJapanese Style Aggregates by Novel:")
    novel_ja = df.groupby(["novel_title", "source"])[ja_cols].mean()
    print(novel_ja.to_string())
else:
    print("\nNo Japanese style columns available.")